# DRS × plastic packaging recycling: country–year panel (v2)

**Question:** Do financial incentives such as a deposit return system (DRS) increase plastic packaging recycling rates?

**What this notebook builds:** one tidy table with one row per country per year (2000–2023), combining
1. **Outcome**: Eurostat plastic packaging recycling rate, 2000–2023 (`cei_wm020_plastic_page_jsonstat.json`)
2. **Optional volumes**: plastic generated / recycled in kg per inhabitant, 2014–2023 (`env_waspac__custom_22962277_jsonstat.json`)
3. **DRS timing**: start years typed in below (Reloop Global Deposit Book 2024, plus the Reloop deck)
4. **DRS intensity**: deposit amount and return rate from the Reloop 2020 and 2022 editions (`deposit_return_schemes_2020_2022.csv`)

**Put the three input files in the `data/` folder** next to this notebook (or change `DATA_DIR` below).

**Read this before analysing.** The deposit amounts exist as two or three dated snapshots per country, not as a yearly series. The notebook carries each snapshot forward and flags every year where it did so (`deposit_is_snapshot_year`). Deposit is known only from the first snapshot year (2018 or 2019) onwards, so analyses that use the deposit amount should be restricted to roughly 2018–2023.

In [1]:
import json
import os
import re
from pathlib import Path

import numpy as np
import pandas as pd

DATA_DIR = Path(os.environ.get("DRS_DATA_DIR", "data"))        # folder with the input files
OUT_DIR = Path(os.environ.get("DRS_OUT_DIR", str(DATA_DIR)))   # where the outputs are written
OUT_DIR.mkdir(parents=True, exist_ok=True)

OUTCOME_FILE = DATA_DIR / "cei_wm020_plastic_page_jsonstat.json"
WASTE_FILE = DATA_DIR / "env_waspac__custom_22962277_jsonstat.json"   # optional (2014-2023 only)
DEPOSIT_CSV = DATA_DIR / "deposit_return_schemes_2020_2022.csv"

AGGREGATES = ["EU27_2020", "EA21", "EA20", "EA19"]   # Eurostat totals, not countries
PLASTIC = "W150102"
RATE_UNIT = "RT_TGT2025"

# Approximate EUR per 1 unit of local currency, used only to make deposits comparable across countries.
# They are rough period averages, NOT sourced. Replace with Eurostat annual averages (table ert_bil_eur_a)
# if the euro value matters for your analysis. National-currency values are also kept in the snapshot table.
FX_EUR_PER_LOCAL = {
    "EUR": 1.0,
    "HRK": 1 / 7.5,
    "DKK": 1 / 7.46,
    "NOK": 1 / 10.2,
    "SEK": 1 / 10.4,
    "ISK": 1 / 147.0,
}

## 1. Read the Eurostat files

In [3]:
def parse_jsonstat(path):
    """Turn a Eurostat JSON-stat 2.0 file into a long DataFrame:
    one column per dimension + 'value' + 'flag' (Eurostat status code)."""
    with open(path, encoding="utf-8") as f:
        d = json.load(f)

    ids, size = d["id"], d["size"]
    codes, labels = [], {}
    for dim in ids:
        cat = d["dimension"][dim]["category"]
        idx = cat["index"]
        order = sorted(idx, key=idx.get) if isinstance(idx, dict) else list(idx)
        codes.append(order)
        labels[dim] = cat.get("label", {})

    n = int(np.prod(size))
    values = np.full(n, np.nan)
    v = d["value"]
    if isinstance(v, dict):                      # sparse form: {flat_index: value}
        for k, x in v.items():
            values[int(k)] = x
    else:                                        # dense form: list with nulls
        values[:] = [np.nan if x is None else x for x in v]

    flags = np.full(n, "", dtype=object)
    s = d.get("status", {})
    if isinstance(s, dict):
        for k, x in s.items():
            flags[int(k)] = x
    elif isinstance(s, list):
        for k, x in enumerate(s):
            flags[k] = x or ""

    index = pd.MultiIndex.from_product(codes, names=ids)
    df = pd.DataFrame({"value": values, "flag": flags}, index=index).reset_index()
    return df, labels


raw, labels = parse_jsonstat(OUTCOME_FILE)
outcome = (
    raw[(raw["waste"] == PLASTIC) & (raw["unit"] == RATE_UNIT)]
    .rename(columns={"value": "plastic_recycling_rate", "flag": "rate_flag"})
    [["geo", "time", "plastic_recycling_rate", "rate_flag"]]
    .assign(year=lambda x: x["time"].astype(int))
    .drop(columns="time")
)
country_names = labels["geo"]
print("outcome rows:", len(outcome), "| years:", outcome["year"].min(), "-", outcome["year"].max())

# Optional: volumes per inhabitant (only exists for 2014-2023)
waste = None
if WASTE_FILE.exists():
    w_raw, _ = parse_jsonstat(WASTE_FILE)
    op_names = {
        "GEN": "plastic_generated_kg_hab",
        "RCY": "plastic_recycled_kg_hab",
        "RCY_NAT": "recycled_in_country_kg_hab",
        "RCY_EU_FOR": "recycled_other_eu_kg_hab",
        "RCY_NEU": "recycled_outside_eu_kg_hab",
    }
    waste = (
        w_raw[(w_raw["waste"] == PLASTIC) & (w_raw["unit"] == "KG_HAB")]
        .assign(year=lambda x: x["time"].astype(int), col=lambda x: x["wst_oper"].map(op_names))
        .pivot_table(index=["geo", "year"], columns="col", values="value", aggfunc="first")
        .reset_index()
    )
    waste.columns.name = None
    print("volume rows:", len(waste))
else:
    print("No volumes file found; volume columns will be left empty.")
outcome.head()

outcome rows: 984 | years: 2000 - 2023
volume rows: 302


,geo,plastic_recycling_rate,rate_flag,year
0,EU27_2020,NaN,,2000
1,EU27_2020,NaN,,2001
2,EU27_2020,NaN,,2002
3,EU27_2020,NaN,,2003
4,EU27_2020,NaN,,2004


## 2. DRS timing (country level)

Start years come from the **Reloop Global Deposit Book 2024** (`drs_start_used`), with the older Reloop deck year kept alongside for comparison (`drs_start_deck`). Where they disagree, `drs_start_check` says so, and you should treat that country with care (Netherlands, Malta, Romania, Portugal).

**Plastic-specific start (`plastic_start_override`).** For the plastic question, what matters is when *plastic* entered the DRS. Finland's system began in 1996 but PET bottles were added in 2008, so Finland counts as treated from 2008. Sweden's 1984 start may have been cans only (PET date not checked); no other plastic dates are overridden.

Countries whose DRS starts after 2023 (HU, IE, AT, PL, PT) are listed so they are correctly treated as *not yet treated*.

In [4]:
drs = pd.DataFrame(
    [
        # geo, start (GDB 2024), start (deck), plastic start override, note
        ("SE", 1984, 1984, None, "Possibly cans first; PET start year not checked"),
        ("IS", 1989, 1989, None, "Deposit increases 2002, 2010, Jan 2021, Jan 2023"),
        ("FI", 1996, 1996, 2008, "PET bottles added 2008, glass 2012 (GDB 2024)"),
        ("NO", 1999, 1999, None, "Deposit increased in 2018"),
        ("DK", 2002, 2002, None, "Scope expanded 2005, 2008, 2020"),
        ("DE", 2003, 2003, None, "Scope expanded 2022 and 2024"),
        ("EE", 2005, 2005, None, "Deposit increases June 2010, July 2011, Feb 2015"),
        ("NL", 2007, 2005, None, "Deck says 2005, GDB 2024 says 2007. Large PET only at first; small PET 2021; cans 2023"),
        ("HR", 2006, 2006, None, "0.50 HRK until 2022; EUR 0.07 from 2023"),
        ("LT", 2016, 2016, None, "In-window switcher"),
        ("MT", 2022, 2020, None, "Deck says 2020, GDB 2024 says 2022"),
        ("SK", 2022, 2022, None, "In-window switcher"),
        ("LV", 2022, 2022, None, "Started 1 Feb 2022"),
        ("RO", 2023, 2022, None, "Deck says 2022, GDB 2024 says 2023 (Nov 2023)"),
        ("PT", 2026, 2022, None, "Deck says 2022; GDB 2024 lists it as upcoming (2026)"),
        ("HU", 2024, None, None, "Outside the data window"),
        ("IE", 2024, None, None, "Outside the data window"),
        ("AT", 2025, None, None, "Outside the data window"),
        ("PL", 2025, None, None, "Planned for 2025, possible delay; outside the data window"),
    ],
    columns=["geo", "drs_start_used", "drs_start_deck", "plastic_start_override", "drs_note"],
)
drs["plastic_drs_start"] = drs["plastic_start_override"].fillna(drs["drs_start_used"])

def check(row):
    if pd.isna(row["drs_start_deck"]):
        return "not in deck"
    return "agrees with deck" if row["drs_start_used"] == row["drs_start_deck"] else "DIFFERS from deck"

drs["drs_start_check"] = drs.apply(check, axis=1)
drs = drs.drop(columns="plastic_start_override")
drs

,geo,drs_start_used,drs_start_deck,drs_note,plastic_drs_start,drs_start_check
0,SE,1984,1984.0,Possibly cans first; PET start year not checked,1984.0,agrees with deck
1,IS,1989,1989.0,"Deposit increases 2002, 2010, Jan 2021, Jan 2023",1989.0,agrees with deck
2,FI,1996,1996.0,"PET bottles added 2008, glass 2012 (GDB 2024)",2008.0,agrees with deck
3,NO,1999,1999.0,Deposit increased in 2018,1999.0,agrees with deck
4,DK,2002,2002.0,"Scope expanded 2005, 2008, 2020",2002.0,agrees with deck
5,DE,2003,2003.0,Scope expanded 2022 and 2024,2003.0,agrees with deck
6,EE,2005,2005.0,"Deposit increases June 2010, July 2011, Feb 2015",2005.0,agrees with deck
7,NL,2007,2005.0,"Deck says 2005, GDB 2024 says 2007. Large PET ...",2007.0,DIFFERS from deck
8,HR,2006,2006.0,0.50 HRK until 2022; EUR 0.07 from 2023,2006.0,agrees with deck
9,LT,2016,2016.0,In-window switcher,2016.0,agrees with deck


## 3. DRS intensity: deposit and return-rate snapshots (Reloop 2020 and 2022 editions)

The CSV covers deposit systems worldwide. Only the European ones can be matched to Eurostat, so the others are dropped. Each edition gives one dated snapshot per country.

How the messy text columns are handled:
- **Refund value** is parsed into local currency amounts and converted to euro with the approximate rates in the config cell. The 2020 edition gives one range per country (all materials); the 2022 edition lists amounts by material and size, so a plastic-only minimum is available for 2022 only.
- **Main intensity measure: `deposit_max_eur`**, the highest deposit tier (usually the large bottle; the only amount in flat systems). It is the one measure that means the same thing in both editions. The lowest tier is *not* comparable across editions (e.g. Denmark's 1 DKK in 2020 is for metal/glass; the plastic minimum is 1.5 DKK).
- **Return rate** is the share of deposit containers returned (not a recycling rate). The text field is parsed only when it is a single overall percentage.
- If a row has no year (Latvia, Malta, Slovakia in the 2022 edition), the snapshot is dated to the DRS start year and flagged.

In [5]:
NAME2GEO = {
    "Croatia": "HR", "Denmark": "DK", "Estonia": "EE", "Finland": "FI", "Germany": "DE",
    "Iceland": "IS", "Latvia": "LV", "Lithuania": "LT", "Malta": "MT", "Netherlands": "NL",
    "The Netherlands": "NL", "Norway": "NO", "Slovakia": "SK", "Sweden": "SE",
}

def parse_year(v):
    m = re.match(r"\s*(\d{4})", str(v))
    return int(m.group(1)) if m else np.nan

def parse_rate(t):
    # keep only plain "89%"-style entries; "Plastic <1L: 57%; ..." has no single overall rate
    m = re.match(r"\s*(\d+(?:\.\d+)?)\s*%", str(t))
    return float(m.group(1)) if m else np.nan

def find_amounts(segment):
    vals = []
    for m in re.finditer(r"(?<![\d.])(\d+(?:\.\d+)?)\s*(HRK|DKK|ISK|NOK|SEK)\b", segment):
        vals.append((float(m.group(1)), m.group(2)))
    for m in re.finditer(r"€\s*(\d+(?:\.\d+)?)", segment):
        vals.append((float(m.group(1)), "EUR"))
    return vals

def parse_refund(text):
    out = dict(currency=None, deposit_min_local=np.nan, deposit_max_local=np.nan,
               plastic_min_local=np.nan, deposit_structure=None)
    if pd.isna(text):
        return out
    t = str(text)
    main = re.sub(r"\([^)]*\)", " ", t)          # drop bracketed conversions and "(VAT incl.)"
    vals = find_amounts(main)
    if not vals:
        return out
    out["currency"] = vals[0][1]
    nums = [v for v, c in vals if c == out["currency"]]
    out["deposit_min_local"], out["deposit_max_local"] = min(nums), max(nums)
    if t.strip().startswith("Fixed"):
        out["deposit_structure"] = "flat"
    elif t.strip().startswith("Variable"):
        out["deposit_structure"] = "variable"
    else:
        out["deposit_structure"] = "variable (range)" if min(nums) != max(nums) else "flat"
    if t.strip().startswith("Variable"):             # 2022 edition lists amounts by material/size
        pl = [s for s in main.split(";") if "plastic" in s.lower()]
        pl_nums = [v for s in pl for v, c in find_amounts(s) if c == out["currency"]]
        if pl_nums:
            out["plastic_min_local"] = min(pl_nums)
    return out

csv = pd.read_csv(DEPOSIT_CSV, encoding="utf-8-sig")
csv.columns = ["region", "year_raw", "scope_text", "refund_text", "return_rate_text", "mechanism", "edition"]
csv["geo"] = csv["region"].map(NAME2GEO)
eu = csv[csv["geo"].notna()].copy()
print(f"kept {len(eu)} of {len(csv)} rows (non-European systems dropped); countries: {sorted(eu['geo'].unique())}")

parsed = pd.DataFrame([parse_refund(t) for t in eu["refund_text"]], index=eu.index)
snap = pd.concat([eu, parsed], axis=1)
snap["snapshot_year"] = snap["year_raw"].apply(parse_year)

# rows without a year: date them to the DRS start year and flag
start_map = drs.set_index("geo")["plastic_drs_start"]
snap["snapshot_year_imputed"] = snap["snapshot_year"].isna()
snap.loc[snap["snapshot_year_imputed"], "snapshot_year"] = snap.loc[snap["snapshot_year_imputed"], "geo"].map(start_map)
snap["snapshot_year"] = snap["snapshot_year"].astype(int)

fx = snap["currency"].map(FX_EUR_PER_LOCAL)
snap["deposit_min_eur"] = snap["deposit_min_local"] * fx
snap["deposit_max_eur"] = snap["deposit_max_local"] * fx
snap["deposit_plastic_min_eur"] = snap["plastic_min_local"] * fx
snap["return_rate_pct"] = snap["return_rate_text"].apply(parse_rate)
snap["covers_plastic"] = snap["scope_text"].str.contains("plastic", case=False)
snap["covers_metal"] = snap["scope_text"].str.contains("metal", case=False)
snap["covers_glass"] = snap["scope_text"].str.contains("glass", case=False)
snap["edition"] = "Reloop " + snap["edition"].astype(str)

# Values read from the Global Deposit Book 2024 that extend the CSV (add more rows here if you find them)
manual = pd.DataFrame(
    [
        # geo, year, currency, max_local, edition note
        ("IS", 2023, "ISK", 20.0, "Reloop GDB 2024: flat ISK 20 after the January 2023 increase"),
    ],
    columns=["geo", "snapshot_year", "currency", "deposit_max_local", "refund_text"],
)
manual["deposit_min_local"] = manual["deposit_max_local"]
manual["edition"] = "Reloop GDB 2024 (manual)"
manual["deposit_structure"] = "flat"
manual["snapshot_year_imputed"] = False
mfx = manual["currency"].map(FX_EUR_PER_LOCAL)
manual["deposit_min_eur"] = manual["deposit_min_local"] * mfx
manual["deposit_max_eur"] = manual["deposit_max_local"] * mfx
snap = pd.concat([snap, manual], ignore_index=True)

snap_cols = ["geo", "edition", "snapshot_year", "snapshot_year_imputed", "deposit_structure", "currency",
             "deposit_min_local", "deposit_max_local", "deposit_min_eur", "deposit_max_eur",
             "deposit_plastic_min_eur", "return_rate_pct", "return_rate_text",
             "covers_plastic", "covers_metal", "covers_glass", "mechanism", "scope_text", "refund_text"]
snap = snap[snap_cols].sort_values(["geo", "snapshot_year"]).reset_index(drop=True)
snap

kept 23 of 96 rows (non-European systems dropped); countries: ['DE', 'DK', 'EE', 'FI', 'HR', 'IS', 'LT', 'LV', 'MT', 'NL', 'NO', 'SE', 'SK']


,geo,edition,snapshot_year,snapshot_year_imputed,deposit_structure,currency,deposit_min_local,deposit_max_local,deposit_min_eur,deposit_max_eur,deposit_plastic_min_eur,return_rate_pct,return_rate_text,covers_plastic,covers_metal,covers_glass,mechanism,scope_text,refund_text
0,DE,Reloop 2020,2018,False,flat,EUR,0.25,0.25,0.250000,0.250000,NaN,97.0,97% (2018),True,True,True,Return-to-retail,"Plastic, metal, glass (containers 100ml-3L)",€0.25 (USD$0.28)
1,DE,Reloop 2022,2021,False,flat,EUR,0.25,0.25,0.250000,0.250000,NaN,NaN,NaN,True,True,True,Return-to-retail,"Plastic, metal, glass",Fixed: €0.25
2,DK,Reloop 2020,2019,False,variable (range),DKK,1.00,3.00,0.134048,0.402145,NaN,94.0,94% (2019),True,True,True,Return-to-retail,"Plastic, metal, glass","1 DKK (€0.13, USD$0.15) to 3 DKK (€0.40, USD$0..."
3,DK,Reloop 2022,2021,False,variable,DKK,1.00,3.00,0.134048,0.402145,0.201072,95.0,95%,True,True,True,Return-to-retail,"Plastic, metal, glass","Variable: Metal, glass <1L: 1 DKK; Plastic <1L..."
4,EE,Reloop 2020,2019,False,flat,EUR,0.10,0.10,0.100000,0.100000,NaN,90.0,90% (2019),True,True,True,Return-to-retail,"Plastic, metal, glass (containers 100ml-3L)",€0.10 (USD$0.11)
5,EE,Reloop 2022,2020,False,flat,EUR,0.10,0.10,0.100000,0.100000,NaN,91.0,91%,True,True,True,Return-to-retail,"Plastic, metal, glass",Fixed: €0.10
6,FI,Reloop 2020,2019,False,variable (range),EUR,0.10,0.40,0.100000,0.400000,NaN,90.0,90% (2019),True,True,True,Return-to-retail,"Plastic, metal, glass",€0.10 (USD$0.11) to €0.40 (USD$0.45)
7,FI,Reloop 2022,2021,False,variable,EUR,0.10,0.40,0.100000,0.400000,0.100000,90.0,90%,True,True,True,Return-to-retail,"Plastic, metal, glass",Variable: Plastic ≤350ml: €0.10; Plastic 351ml...
8,HR,Reloop 2020,2019,False,flat,HRK,0.50,0.50,0.066667,0.066667,NaN,89.0,89% (2019),True,True,True,Return-to-retail,"Plastic, metal, glass (containers >200ml)","0.50 HRK (€0.06, USD$0.07)"
9,HR,Reloop 2022,2020,False,flat,HRK,0.50,0.50,0.066667,0.066667,NaN,88.0,88%,True,True,True,Return-to-retail,"Plastic, metal, glass",Fixed: 0.50 HRK


In [6]:
# Quick look: does the deposit change within countries between snapshots?
chg = (snap.groupby("geo")["deposit_max_local"].agg(["min", "max", "count"])
          .assign(changed=lambda d: d["min"] != d["max"]))
chg

,min,max,count,changed
geo,,,,
DE,0.25,0.25,2,False
DK,3.00,3.00,2,False
EE,0.10,0.10,2,False
FI,0.40,0.40,2,False
HR,0.50,0.50,2,False
IS,16.00,20.00,3,True
LT,0.10,0.10,2,False
LV,0.10,0.10,1,False
MT,0.10,0.10,1,False


## 4. Assemble the country–year panel

In [7]:
geos = [g for g in country_names if g not in AGGREGATES]
years = sorted(outcome["year"].unique())

panel = pd.MultiIndex.from_product([geos, years], names=["geo", "year"]).to_frame(index=False)
panel["country"] = panel["geo"].map(country_names)
panel = panel.merge(outcome, on=["geo", "year"], how="left")
if waste is not None:
    panel = panel.merge(waste, on=["geo", "year"], how="left")
else:
    for c in ["plastic_generated_kg_hab", "plastic_recycled_kg_hab", "recycled_in_country_kg_hab",
              "recycled_other_eu_kg_hab", "recycled_outside_eu_kg_hab"]:
        panel[c] = np.nan

# Drop geographies with no outcome at all (e.g. CH, UK, Western Balkans, TR in this extract)
has_data = panel.groupby("geo")["plastic_recycling_rate"].apply(lambda s: s.notna().any())
dropped = has_data[~has_data].index.tolist()
panel = panel[panel["geo"].isin(has_data[has_data].index)].copy()
print("dropped (no outcome data):", dropped)

panel = panel.merge(drs, on="geo", how="left")

# --- DRS timing variables (plastic-specific) ------------------------------------------
panel["has_drs"] = (panel["year"] >= panel["plastic_drs_start"]).fillna(False).astype(int)
panel["drs_full_year"] = (panel["year"] > panel["plastic_drs_start"]).fillna(False).astype(int)
in_window = panel["plastic_drs_start"].notna() & (panel["plastic_drs_start"] <= years[-1])
panel["years_since_drs"] = np.where(in_window, panel["year"] - panel["plastic_drs_start"], np.nan)

def drs_group(start):
    if pd.isna(start) or start > years[-1]:
        return "3_no_drs_in_window"
    if start <= years[0]:
        return "1_drs_before_window"
    return "2_drs_introduced_in_window"

panel["drs_group"] = panel["plastic_drs_start"].apply(drs_group)

# --- DRS intensity: carry each snapshot forward until the next one ----------------------
snap_asof = (snap.dropna(subset=["deposit_max_eur"])
                 .drop_duplicates(["geo", "snapshot_year"], keep="last")
                 .sort_values("snapshot_year"))
snap_asof = snap_asof[["geo", "snapshot_year", "deposit_max_eur", "deposit_min_eur", "deposit_plastic_min_eur",
                       "deposit_structure", "covers_metal", "covers_glass"]]
panel = panel.sort_values("year")
panel = pd.merge_asof(panel, snap_asof, left_on="year", right_on="snapshot_year", by="geo",
                      direction="backward", suffixes=("", "_snap"))
panel = panel.rename(columns={"snapshot_year": "deposit_source_year"})

no_drs = panel["has_drs"] == 0
for c in ["deposit_max_eur", "deposit_min_eur", "deposit_plastic_min_eur"]:
    panel.loc[no_drs, c] = 0.0                       # no DRS in force = no deposit
panel.loc[no_drs, ["deposit_structure", "covers_metal", "covers_glass", "deposit_source_year"]] = np.nan
panel["deposit_is_snapshot_year"] = panel["deposit_source_year"] == panel["year"]

# return rate only where a snapshot exists for that exact year
rr = (snap.dropna(subset=["return_rate_pct"])[["geo", "snapshot_year", "return_rate_pct"]]
          .rename(columns={"snapshot_year": "year"}).drop_duplicates(["geo", "year"]))
panel = panel.merge(rr, on=["geo", "year"], how="left")

# --- measurement flags -------------------------------------------------------------------
panel["series_break_flag"] = panel["rate_flag"].isin(["b", "bd", "be"])
panel["post_2020_method_period"] = (panel["year"] >= 2020).astype(int)
panel["rate_recomputed"] = panel["plastic_recycled_kg_hab"] / panel["plastic_generated_kg_hab"] * 100
abroad = panel["recycled_other_eu_kg_hab"] + panel["recycled_outside_eu_kg_hab"]
panel["share_recycled_abroad_pct"] = abroad / panel["plastic_recycled_kg_hab"] * 100

panel = panel.sort_values(["geo", "year"]).reset_index(drop=True)
panel.head(10)

dropped (no outcome data): ['AL', 'CH', 'ME', 'MK', 'RS', 'TR', 'UK']


,geo,year,country,plastic_recycling_rate,rate_flag,plastic_generated_kg_hab,plastic_recycled_kg_hab,recycled_in_country_kg_hab,recycled_other_eu_kg_hab,recycled_outside_eu_kg_hab,...,deposit_plastic_min_eur,deposit_structure,covers_metal,covers_glass,deposit_is_snapshot_year,return_rate_pct,series_break_flag,post_2020_method_period,rate_recomputed,share_recycled_abroad_pct
0,AT,2000,Austria,26.2,,NaN,NaN,NaN,NaN,NaN,...,0.0,NaN,NaN,NaN,False,NaN,False,0,NaN,NaN
1,AT,2001,Austria,29.1,,NaN,NaN,NaN,NaN,NaN,...,0.0,NaN,NaN,NaN,False,NaN,False,0,NaN,NaN
2,AT,2002,Austria,30.0,,NaN,NaN,NaN,NaN,NaN,...,0.0,NaN,NaN,NaN,False,NaN,False,0,NaN,NaN
3,AT,2003,Austria,30.7,,NaN,NaN,NaN,NaN,NaN,...,0.0,NaN,NaN,NaN,False,NaN,False,0,NaN,NaN
4,AT,2004,Austria,33.2,,NaN,NaN,NaN,NaN,NaN,...,0.0,NaN,NaN,NaN,False,NaN,False,0,NaN,NaN
5,AT,2005,Austria,32.9,,NaN,NaN,NaN,NaN,NaN,...,0.0,NaN,NaN,NaN,False,NaN,False,0,NaN,NaN
6,AT,2006,Austria,35.8,,NaN,NaN,NaN,NaN,NaN,...,0.0,NaN,NaN,NaN,False,NaN,False,0,NaN,NaN
7,AT,2007,Austria,32.7,,NaN,NaN,NaN,NaN,NaN,...,0.0,NaN,NaN,NaN,False,NaN,False,0,NaN,NaN
8,AT,2008,Austria,34.9,,NaN,NaN,NaN,NaN,NaN,...,0.0,NaN,NaN,NaN,False,NaN,False,0,NaN,NaN
9,AT,2009,Austria,32.7,,NaN,NaN,NaN,NaN,NaN,...,0.0,NaN,NaN,NaN,False,NaN,False,0,NaN,NaN


## 5. Sanity checks

In [8]:
assert not panel.duplicated(["geo", "year"]).any()
print("rows:", len(panel), "| countries:", panel["geo"].nunique(), "| years:", years[0], "-", years[-1])

# 1. Official rate vs recomputed from volumes (2014-2023 only)
diff = (panel["plastic_recycling_rate"] - panel["rate_recomputed"]).abs()
print(f"official vs recomputed rate: median {diff.median():.2f} pp, max {diff.max():.2f} pp (n={diff.notna().sum()})")

# 2. Rates above 100% would signal a measurement problem
print("rates > 100:", int((panel["plastic_recycling_rate"] > 100).sum()))

# 3. Missing outcome
miss = panel[panel["plastic_recycling_rate"].isna()].groupby("geo")["year"].apply(list)
print("country-years with no recycling rate:", int(panel["plastic_recycling_rate"].isna().sum()))
display(miss)

# 4. How many pre/post years does each in-window DRS country actually have?
sw = panel[panel["drs_group"] == "2_drs_introduced_in_window"].copy()
sw["pre"] = (sw["has_drs"] == 0) & sw["plastic_recycling_rate"].notna()
sw["post"] = (sw["has_drs"] == 1) & sw["plastic_recycling_rate"].notna()
ident = sw.groupby("geo").agg(plastic_drs_start=("plastic_drs_start", "first"),
                              pre_years=("pre", "sum"), post_years=("post", "sum"),
                              check=("drs_start_check", "first"))
print("DRS introduced inside 2000-2023 (pre/post years with an outcome):")
display(ident)

# 5. Deposit coverage among country-years with a DRS in force
d = panel[panel["has_drs"] == 1]
print(f"DRS country-years: {len(d)} | with a deposit value: {d['deposit_max_eur'].notna().sum()}"
      f" | of which carried forward from an earlier snapshot: "
      f"{(d['deposit_max_eur'].notna() & ~d['deposit_is_snapshot_year']).sum()}")
display(d.dropna(subset=["deposit_max_eur"]).groupby("geo")["year"].agg(["min", "max"]))

# 6. Group sizes
display(panel.groupby("drs_group")["geo"].nunique().rename("n_countries"))

rows: 720 | countries: 30 | years: 2000 - 2023
official vs recomputed rate: median 0.02 pp, max 0.07 pp (n=292)
rates > 100: 0
country-years with no recycling rate: 90


geo
BG           [2000, 2001, 2002, 2003, 2004, 2020, 2023]
CY                             [2000, 2001, 2002, 2003]
CZ                                   [2000, 2001, 2002]
EE                             [2000, 2001, 2002, 2003]
EL                                               [2020]
HR    [2000, 2001, 2002, 2003, 2004, 2005, 2006, 200...
HU                       [2000, 2001, 2002, 2003, 2023]
IS    [2000, 2001, 2002, 2003, 2004, 2005, 2006, 200...
LI                 [2000, 2001, 2002, 2003, 2004, 2005]
LT                             [2000, 2001, 2002, 2003]
LV                             [2000, 2001, 2002, 2003]
MT                             [2000, 2001, 2002, 2003]
NO                 [2000, 2001, 2002, 2003, 2004, 2005]
PL           [2000, 2001, 2002, 2003, 2020, 2021, 2023]
RO                       [2000, 2001, 2002, 2003, 2004]
SI                             [2000, 2001, 2002, 2003]
SK                                   [2000, 2001, 2002]
Name: year, dtype: object

DRS introduced inside 2000-2023 (pre/post years with an outcome):


,plastic_drs_start,pre_years,post_years,check
geo,,,,
DE,2003.0,3,21,agrees with deck
DK,2002.0,2,22,agrees with deck
EE,2005.0,1,19,agrees with deck
FI,2008.0,8,16,agrees with deck
HR,2006.0,0,11,agrees with deck
LT,2016.0,12,8,agrees with deck
LV,2022.0,18,2,agrees with deck
MT,2022.0,18,2,DIFFERS from deck
NL,2007.0,7,17,DIFFERS from deck


DRS country-years: 200 | with a deposit value: 58 | of which carried forward from an earlier snapshot: 34


,min,max
geo,,
DE,2018,2023
DK,2019,2023
EE,2019,2023
FI,2019,2023
HR,2019,2023
IS,2019,2023
LT,2019,2023
LV,2022,2023
MT,2022,2023


drs_group
1_drs_before_window            3
2_drs_introduced_in_window    11
3_no_drs_in_window            16
Name: n_countries, dtype: int64

## 6. Save

In [9]:
column_order = [
    "geo", "country", "year",
    # outcome
    "plastic_recycling_rate", "rate_flag", "series_break_flag", "post_2020_method_period",
    "plastic_generated_kg_hab", "plastic_recycled_kg_hab", "share_recycled_abroad_pct",
    # DRS timing
    "has_drs", "drs_full_year", "years_since_drs", "drs_group",
    "plastic_drs_start", "drs_start_used", "drs_start_deck", "drs_start_check", "drs_note",
    # DRS intensity
    "deposit_max_eur", "deposit_min_eur", "deposit_plastic_min_eur", "deposit_structure",
    "deposit_source_year", "deposit_is_snapshot_year", "covers_metal", "covers_glass", "return_rate_pct",
]
final = panel[column_order]

dictionary = pd.DataFrame(
    [
        ("geo", "Eurostat geo code (2 letters)", "Eurostat"),
        ("country", "Country name", "Eurostat"),
        ("year", "Calendar year, 2000-2023", "Eurostat"),
        ("plastic_recycling_rate", "Plastic packaging recycling rate, % (2025-target basis). OUTCOME", "Eurostat CEI_WM020 / ENV_WASPACR"),
        ("rate_flag", "Eurostat status flag (b/bd/be = break in series, d = definition differs, e = estimated, p = provisional, i = imputed)", "Eurostat"),
        ("series_break_flag", "True if rate_flag signals a break in the time series", "derived"),
        ("post_2020_method_period", "1 for 2020 onwards. Many countries drop in 2020, probably a reporting-method change (check Eurostat metadata)", "derived"),
        ("plastic_generated_kg_hab", "Plastic packaging waste generated, kg per inhabitant (2014-2023 only)", "Eurostat ENV_WASPAC"),
        ("plastic_recycled_kg_hab", "Plastic packaging waste recycled, kg per inhabitant (2014-2023 only)", "Eurostat ENV_WASPAC"),
        ("share_recycled_abroad_pct", "Share of recycled plastic packaging recycled outside the country, % (partial coverage)", "derived"),
        ("has_drs", "1 if year >= plastic_drs_start, else 0. MAIN EXPLANATORY VARIABLE", "hand-coded"),
        ("drs_full_year", "1 if year > plastic_drs_start (first full year of operation)", "hand-coded"),
        ("years_since_drs", "year - plastic_drs_start; negative = before; empty = no DRS in window", "derived"),
        ("drs_group", "1 = DRS at or before 2000, 2 = introduced 2001-2023, 3 = none by 2023", "derived"),
        ("plastic_drs_start", "Year plastic containers entered the DRS (used for has_drs)", "Reloop GDB 2024"),
        ("drs_start_used", "Year the DRS as a whole started, per the Global Deposit Book 2024", "Reloop GDB 2024"),
        ("drs_start_deck", "Start year in the older Reloop deck", "Reloop deck"),
        ("drs_start_check", "Whether the GDB 2024 year agrees with the deck", "derived"),
        ("drs_note", "Free-text note on the DRS (scope changes, deposit changes, conflicts)", "hand-coded"),
        ("deposit_max_eur", "Highest deposit tier in euro; 0 when no DRS. Snapshot values carried forward. KNOWN ONLY FROM ~2018. Approximate FX", "Reloop 2020/2022 editions"),
        ("deposit_min_eur", "Lowest deposit tier in euro (all materials). Not comparable across editions for variable systems, e.g. Denmark", "Reloop 2020/2022 editions"),
        ("deposit_plastic_min_eur", "Lowest plastic deposit in euro, 2022-edition snapshots onwards only", "Reloop 2022 edition"),
        ("deposit_structure", "flat, variable or variable (range)", "Reloop 2020/2022 editions"),
        ("deposit_source_year", "Year of the snapshot the deposit value comes from", "derived"),
        ("deposit_is_snapshot_year", "True if the deposit value is for this exact year; False = carried forward", "derived"),
        ("covers_metal", "Scope text of the latest snapshot mentions metal", "Reloop 2020/2022 editions"),
        ("covers_glass", "Scope text of the latest snapshot mentions glass", "Reloop 2020/2022 editions"),
        ("return_rate_pct", "Share of deposit containers returned, %. Only in snapshot years. NOT a recycling rate", "Reloop 2020/2022 editions"),
    ],
    columns=["column", "description", "source"],
)
assert set(final.columns) == set(dictionary["column"]), set(final.columns) ^ set(dictionary["column"])

final.to_csv(OUT_DIR / "drs_recycling_panel_v2.csv", index=False, encoding="utf-8")
snap.to_csv(OUT_DIR / "drs_deposit_snapshots.csv", index=False, encoding="utf-8")
dictionary.to_csv(OUT_DIR / "drs_recycling_panel_v2_dictionary.csv", index=False, encoding="utf-8")
print("Saved to", OUT_DIR.resolve(), "|", final.shape)

Saved to /Users/leannericard/mgt499_project/data | (720, 28)


## 7. First look (starting point for the univariate / bivariate work)

In [10]:
final["plastic_recycling_rate"].describe()

count    630.000000
mean      33.609048
std       13.868971
min        1.900000
25%       24.175000
50%       33.050000
75%       42.975000
max       81.700000
Name: plastic_recycling_rate, dtype: float64

In [11]:
# Average recycling rate by DRS group over time
(final.pivot_table(index="year", columns="drs_group", values="plastic_recycling_rate", aggfunc="mean").round(1))

drs_group,1_drs_before_window,2_drs_introduced_in_window,3_no_drs_in_window
year,,,
2000,13.5,25.7,16.5
2001,17.2,25.3,18.4
2002,19.8,23.9,19.4
2003,22.3,23.3,21.2
2004,25.2,18.7,22.1
2005,29.9,20.9,22.7
2006,36.8,26.1,24.2
2007,35.8,27.4,25.5
2008,31.8,27.7,26.8


In [12]:
# Deposit vs recycling rate, DRS country-years with a known deposit (roughly 2018-2023)
dd = final[(final["has_drs"] == 1)].dropna(subset=["deposit_max_eur", "plastic_recycling_rate"])
print("n =", len(dd), "| countries =", dd["geo"].nunique())
dd[["deposit_max_eur", "plastic_recycling_rate"]].corr().round(2)

n = 57 | countries = 13


,deposit_max_eur,plastic_recycling_rate
deposit_max_eur,1.00,-0.27
plastic_recycling_rate,-0.27,1.00


## What this dataset can and cannot answer

**Can**
- **DRS vs no DRS:** a country comparison over 24 years (30 countries), and before/after comparisons for the countries that introduced a plastic DRS inside the window (see the pre/post table in the checks).
- **Dose-response, cross-section only:** whether higher deposits go with higher recycling rates across about 10–13 DRS countries, 2018–2023.

**Cannot (yet)**
- **Other financial incentives** (landfill taxes, packaging taxes, EPR fees, pay-as-you-throw) are not in the data. As written, "financial incentives such as DRS" is operationalised by DRS only. Either narrow the claim to deposit-based incentives or add a variable for another incentive.
- **Deposit changes over time.** Nearly all deposits are unchanged between the 2019 and 2021 snapshots (check the table in section 3). Only Iceland changes, so almost all deposit variation is across countries, not within a country.
- **Drink bottles vs all plastic packaging.** The outcome counts all plastic packaging, so the DRS signal is diluted.
- **Causal claims.** With very few treated countries and short post-periods for 2022 launches, treat before/after results as exploratory.

**Flags to remember:** the 2020 reporting-method break, approximate FX rates, and the disputed start years (NL, MT, RO, PT).